# Stack smoke test — Neo4j + Qdrant on Docker

Checks the two serving containers and, more importantly, the **join between
them**: a Gene node in Neo4j and a function-text record in Qdrant are the same
protein, keyed by UniProt accession. That join is what the MCP toolkit depends
on, and nothing else in the repo exercises it end to end.

Prerequisites: `docker compose up -d qdrant neo4j`, and `.env` pointing at the
ports those actually bound.

In [1]:
import os, sys
sys.path.insert(0, "src/builders/Qdrant")
from dotenv import load_dotenv
load_dotenv()

QDRANT_URL  = os.getenv("QDRANT_URL", "http://localhost:6333")
NEO4J_CONT  = os.getenv("NEO4J_CONTAINER_URI", "bolt://localhost:7689")
MODEL_PATH  = os.getenv("BIOBERT_MODEL_PATH", "EmbeddingModel/biobert-go-retrieval")
print("qdrant:", QDRANT_URL, "| neo4j:", NEO4J_CONT, "| model:", MODEL_PATH)

qdrant: http://localhost:6333 | neo4j: bolt://localhost:7689 | model: EmbeddingModel/biobert-go-retrieval


## 1. Both services reachable

In [2]:
from neo4j import GraphDatabase, RoutingControl
from core.config import open_client

qc = open_client(url=QDRANT_URL)
# auth=None because docker-compose sets NEO4J_AUTH: none. The container is
# bound to 127.0.0.1, which is the only reason that is acceptable.
nc = GraphDatabase.driver(NEO4J_CONT, auth=None)
nc.verify_connectivity()

def cypher(q, **kw):
    recs, _, _ = nc.execute_query(q, routing_=RoutingControl.READ, **kw)
    return [r.data() for r in recs]

print("neo4j   :", cypher("MATCH (n) RETURN count(n) AS nodes")[0],
      cypher("MATCH ()-[r]->() RETURN count(r) AS rels")[0])
print("aliases :", {a.alias_name: a.collection_name for a in qc.get_aliases().aliases})
for name in ("function_records", "function_chunks"):
    print(f"{name:17}:", qc.count(name).count, "points")

/home/yoyo458/kafka-etl-new/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


neo4j   : {'nodes': 122347} {'rels': 602142}
aliases : {'function_chunks': 'function_chunks__v20261006193533', 'function_records': 'function_records__v20261006193533'}
function_records : 10204 points
function_chunks  : 42714 points


## 2. The cross-store join

`Gene.id` in Neo4j is the UniProt accession, and `uniprot_id` is the Qdrant
record key. If these ever drift apart the MCP tools silently return nothing,
so this is the check worth keeping.

In [3]:
from qdrant_client import models

genes = {r["id"] for r in cypher("MATCH (g:Gene) RETURN g.id AS id")}
print(f"Gene nodes in Neo4j        : {len(genes)}")

recs, _ = qc.scroll("function_records", limit=20000, with_payload=["uniprot_id"])
accessions = {p.payload["uniprot_id"] for p in recs}
print(f"records in Qdrant          : {len(accessions)}")

both = genes & accessions
print(f"in BOTH (joinable)         : {len(both)}")
print(f"gene, no function text     : {len(genes - accessions)}   <- expected: UniProt has none")
print(f"text, not a gene node      : {len(accessions - genes)}   <- expected 0")
assert not (accessions - genes), "Qdrant holds accessions with no Gene node -- the stores disagree"

Gene nodes in Neo4j        : 10561
records in Qdrant          : 10204
in BOTH (joinable)         : 10204
gene, no function text     : 357   <- expected: UniProt has none
text, not a gene node      : 0   <- expected 0


## 3. entrez_id round-trip\n\nThe second lookup key, so a caller holding an Entrez id can still reach the record.

In [4]:
with_ez = qc.count("function_records", count_filter=models.Filter(must_not=[
    models.IsEmptyCondition(is_empty=models.PayloadField(key="entrez_id"))])).count
print(f"records carrying entrez_id : {with_ez} / {len(accessions)}")

# Pick one and look it up BY entrez id -- the reverse direction.
sample, _ = qc.scroll("function_records", limit=1,
                      scroll_filter=models.Filter(must_not=[
                          models.IsEmptyCondition(is_empty=models.PayloadField(key="entrez_id"))]),
                      with_payload=["uniprot_id", "entrez_id"])
acc, ez = sample[0].payload["uniprot_id"], sample[0].payload["entrez_id"]
back, _ = qc.scroll("function_records", limit=5, with_payload=["uniprot_id"],
                    scroll_filter=models.Filter(must=[
                        models.FieldCondition(key="entrez_id", match=models.MatchValue(value=ez))]))
print(f"entrez {ez} -> {[p.payload['uniprot_id'] for p in back]}  (expect {acc})")

# And the same id on the graph side.
print("neo4j says :", cypher("MATCH (g:Gene {id:$a}) RETURN g.id AS id, g.entrez_ids AS entrez_ids", a=acc))

records carrying entrez_id : 9727 / 10204
entrez 3892 -> ['O43790']  (expect O43790)


neo4j says : [{'id': 'O43790', 'entrez_ids': ['3892']}]


## 4. Semantic search\n\nFirst run loads BioBERT (~30s). Hits should be topically coherent, not random.

In [5]:
from core.config import load_encoder, load_sparse_encoder
from core.retriever import Retriever

retr = Retriever(qc, load_encoder(MODEL_PATH), load_sparse_encoder())

for q in ["DNA damage response and p53 signaling",
          "mitochondrial fatty acid beta-oxidation"]:
    print(f"\n=== {q}")
    for h in retr.search(q, k=5):
        snip = (h.get("snippets") or [{}])[0].get("text", "")
        print(f"  {h['uniprot_id']:8} entrez={str(h.get('entrez_id','-')):8} "
              f"rec={h['record_score']:.3f} chunk={h['chunk_score']:.3f}")
        print(f"     {snip[:100]}")

This model was created with Sentence Transformers version 6.1.0, but you're using version 6.0.1. Consider updating to the latest version to avoid potential issues.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4225.43it/s]


unpad_inputs=True was set, but the prerequisites for skipping padding are not met. Falling back to padded inputs.



=== DNA damage response and p53 signaling
  O60885   entrez=23476    rec=0.071 chunk=0.548
     Also acts as a regulator of p53/TP53-mediated transcription: following phosphorylation by CK2, recru
  Q96Q15   entrez=23049    rec=0.020 chunk=0.516
     Can phosphorylate p53/TP53 and is required for optimal p53/TP53 activation after cellular exposure t
  O60285   entrez=9891     rec=0.600 chunk=0.500
     May also act as a tumor malignancy-associated factor by promoting tumor invasion and metastasis unde
  Q15018   entrez=23172    rec=0.250 chunk=0.389
     Required for normal induction of p53/TP53 in response to DNA damage (PubMed:25283148). Independent o
  Q13330   entrez=9112     rec=0.175 chunk=0.340
     Regulates p53-dependent and -independent DNA repair processes following genotoxic stress (PubMed:198

=== mitochondrial fatty acid beta-oxidation


  Q99714   entrez=3028     rec=0.190 chunk=1.000
     Acts as (S)-3-hydroxyacyl-CoA dehydrogenase in mitochondrial fatty acid beta-oxidation, a major degr
  O43772   entrez=788      rec=0.106 chunk=0.348
     Key player in the mitochondrial oxidation pathway, it translocates the fatty acids in the form of ac
  P40939   entrez=3030     rec=0.205 chunk=0.342
     Mitochondrial trifunctional enzyme catalyzes the last three of the four reactions of the mitochondri
  Q16836   entrez=3033     rec=0.533 chunk=0.310
     Mitochondrial fatty acid beta-oxidation enzyme that catalyzes the third step of the beta-oxidation c
  P16219   entrez=35       rec=0.424 chunk=0.290
     Short-chain specific acyl-CoA dehydrogenase is one of the acyl-CoA dehydrogenases that catalyze the 


## 5. Graph-grounded retrieval\n\nThe thing neither store can do alone: start from GO annotations in the graph, end at function text in the vector store.

In [6]:
TERM = "GO_0006281"   # DNA repair

annotated = [r["id"] for r in cypher(
    "MATCH (g:Gene)-[]->(t:Ontology {id:$t}) RETURN g.id AS id", t=TERM)]
print(f"{TERM}: {len(annotated)} annotated genes in the graph")

reachable = [a for a in annotated if a in accessions]
print(f"  with function text in Qdrant: {len(reachable)}")

hits = retr.search("DNA repair", k=20)
overlap = [h["uniprot_id"] for h in hits if h["uniprot_id"] in set(annotated)]
print(f"  of the top-20 for 'DNA repair', {len(overlap)} are annotated to {TERM}:")
print("   ", overlap)

GO_0006281: 186 annotated genes in the graph
  with function text in Qdrant: 184


  of the top-20 for 'DNA repair', 13 are annotated to GO_0006281:
    ['Q92889', 'P07992', 'P09874', 'P15927', 'Q9UQ84', 'Q9NVI1', 'Q14191', 'P39748', 'P78549', 'P35244', 'P49916', 'O43502', 'P27694']


In [7]:
qc.close(); nc.close()
print("ok")

ok
